# GNSS Paper 1 - core pipeline rerun (min-max unification)

Runs `run_pipeline.py`: retrains all 13 detectors from scratch, then the full attack suite and stats/figures, end to end:

`01_classical_baseline -> 02_deep_learning_baseline -> 12_operating_point -> 13_blackbox_attacks -> 14_multisurrogate_transfer -> 18_full_eval -> 17_latency -> 20_mono -> 19_final_analysis -> 22_mcnemar -> 21_figures -> 16_revision_figures`

**Why this rerun is needed:** the feature scaler changed from `StandardScaler` to `MinMaxScaler` so every attack budget (FGSM/PGD/domain/decision-based) is measured in the same `[0,1]` space. Every previously saved model was trained under the old scaler and is now stale -- running any attack/eval script without first retraining 01+02 would silently feed `[0,1]`-range input to a model that expects roughly zero-mean, unit-variance input, producing wrong (not merely different) numbers. This notebook retrains everything from scratch, so that problem cannot occur.

This is a separate notebook from `kaggle_generalization.ipynb`, which only covers `23_generalization.py` and `24_defense.py` (the leave-one-scenario-out / leave-PRN and diagnostic-defense experiments) and does not touch the core 13-detector training or the main adversarial-evaluation numbers (Table 1, the fragility/rank-inversion/trade-off figures).

## Run it overnight WITHOUT losing the result
Use **Save Version -> Save & Run All (Commit)** (top-right), NOT the interactive Run. This whole pipeline is much lighter than experiment 23 (it trains each detector once, not per fold), so it should finish well inside Kaggle's 12h session cap in a single commit -- expect well under 2h on a P100, most of it GridSearchCV in stage 01 and the 3-seed DL training in stage 02.

**Before you commit, set in the right panel:** Accelerator = **GPU**, Internet = **On**, and **Add Input** = your dataset with `texbat_track_combined.csv`.

## 1. Clone the code (Paper-1 branch)

In [ ]:
import os, subprocess, sys
REPO   = "https://github.com/Ojerinde/Master_Research.git"
BRANCH = "paper1-experiment"
DST    = "/kaggle/working/repo"
# Private repo? REPO = "https://<GITHUB_TOKEN>@github.com/Ojerinde/Master_Research.git"
if not os.path.exists(DST):
    subprocess.run(["git","clone","--depth","1","-b",BRANCH,REPO,DST], check=True)
os.chdir(DST)
print("cwd:", os.getcwd()); print("top-level:", sorted(os.listdir("."))[:20])

## 2. Place the corpus CSV where the loader expects it

In [ ]:
import glob, shutil, os
src = glob.glob("/kaggle/input/**/texbat_track_combined.csv", recursive=True)
assert src, "Attach the Kaggle dataset that contains texbat_track_combined.csv (right panel > Add Input)."
os.makedirs("data/processed", exist_ok=True)
shutil.copy(src[0], "data/processed/texbat_track_combined.csv")
print(f"CSV placed: {os.path.getsize('data/processed/texbat_track_combined.csv'):,} bytes")

## 3. Environment check (do NOT `pip install -r requirements.txt` here)

In [ ]:
import sys, subprocess, importlib, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY"))
if not torch.cuda.is_available():
    print("[WARN] No GPU detected. Set Accelerator = GPU in the right panel, then re-run.")
for mod, pip_name in [("imblearn","imbalanced-learn"),("xgboost","xgboost"),("lightgbm","lightgbm"),("sklearn","scikit-learn")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=True)
print("deps OK")

## 4. Run the core pipeline (01 -> 16, the ~1-2h GPU job)

Streams every stage's output live so an error surfaces immediately rather than after the fact. Fails fast: `run_pipeline.py` stops at the first stage that returns nonzero.

In [ ]:
import os, subprocess, sys, time
env = dict(os.environ, PYTHONPATH=".", PYTHONWARNINGS="ignore")
cmd = [sys.executable, "-u", "run_pipeline.py"]
print("running:", " ".join(cmd))
t0 = time.time()
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print(f"[pipeline done] exit={p.returncode}  elapsed={(time.time()-t0)/60:.1f} min")
assert p.returncode == 0, f"run_pipeline.py failed with exit {p.returncode} (scroll up for the traceback)."

## 5. Package everything for download (models + tables + figures)

Zips the retrained models and every result table/figure into one archive, and prints the headline CSVs as text so the key numbers are recoverable even if a file transfer fails.

In [ ]:
import shutil, os
shutil.make_archive("/kaggle/working/results_minmax", "zip", "results")
sz = os.path.getsize("/kaggle/working/results_minmax.zip")
print(f"Wrote /kaggle/working/results_minmax.zip ({sz/1e6:.1f} MB) -- contains results/models, results/tables, results/figures.")

In [ ]:
import pandas as pd, os
pd.set_option('display.max_rows', None); pd.set_option('display.width', 200)
HEADLINE = [
    'results/tables/adversarial_full_oppoint.csv',   # Table 1 / FGSM,PGD,DLSA,SNA,TPA at the operating point
    'results/tables/blackbox_boundary_all.csv',       # decision-based ASR/median-min-Linf per detector
    'results/tables/blackbox_boundary_persample.csv', # per-sample min-Linf -> fragility bootstrap CIs
]
for name in HEADLINE:
    if not os.path.exists(name):
        print(f'[missing] {name}'); continue
    g = pd.read_csv(name)
    print('='*70); print(f'{name}  rows={len(g)}'); print('='*70)
    print(g.round(4).to_string(index=False)[:4000])
    print(f'----BEGIN {os.path.basename(name)}----'); print(g.to_csv(index=False)); print(f'----END {os.path.basename(name)}----')
print('Full results in /kaggle/working/results_minmax.zip -- download from the committed version Output tab.')

### After it finishes
Download `results_minmax.zip` from the committed version's **Output** tab, unzip it over `code/gnss_adversarial_research/results/` locally (or in a fresh clone), and send it back. It replaces every stale (pre-min-max) model and table:
- `results/models/classical/*.joblib`, `results/models/deep_learning/*.pt` -- the 13 retrained detectors.
- `results/tables/*.csv` -- everything the manuscript's numbers come from, including `adversarial_full_oppoint.csv` (Table 1's FGSM/PGD/domain-attack rows) and the `blackbox_boundary_*` files (the decision-based fragility ranking, Figure 6/7).
- `results/figures/*` -- regenerated PDFs/PNGs, ready to copy into `papers/paper1-satnav/figures/` (or rerun `make_figures.py` there against the new tables).